# Chapter 5 revised MLP and GATv2 training

**Status:** canonical revised-method experiment  
**Thesis sections:** 5.1.3–5.2

The revised experiment narrows the model comparison to a wide MLP and GATv2, changes the target to AFC-BJK, and evaluates checkpoints by their FEM objective. It is therefore a new experiment, not a continuation of Chapter 4 training.

## 1. What changed after Chapter 4

Only interior cells contribute to the supervised loss, because boundary values are prescribed. The loss combines Huber and L2 terms against the optimized SUPG parameters, while checkpoint selection uses the physically meaningful AFC-BJK FEM loss—not the training loss alone.

In [ ]:
from supgml.models import RevisedGATv2, RevisedMLP, combined_supervised_loss

# mlp = RevisedMLP(in_channels=graph.x0.shape[1])
# gatv2 = RevisedGATv2(in_channels=graph.x0.shape[1])
# prediction = mlp(graph.x0[graph.interior_dofs])
# loss = combined_supervised_loss(prediction, graph.y_optimized_z[graph.interior_dofs])


## 2. Match the revised model capacity

The revised MLP follows the wide block structure described in Section 5.1.3, with layers up to width 256. Revised GATv2 uses four 256-channel graph-attention layers followed by the same two prediction heads. Because this experiment is restricted to triangular meshes, it uses nine features and omits the Chapter 4 cell-type indicator.

## 3. Select through the PDE

The full FEM comparison is sampled periodically because it entails a state and adjoint solve. Cosine warm restarts schedule the optimizer, and the best saved model is the one with the lowest sampled FEM loss against the AFC-BJK target. The thesis ran MLP for 200,000 epochs and the more expensive GATv2 for 150,000; `epochs_by_architecture` records that asymmetry explicitly.

**Provenance note:** Equation (28) in the thesis prints a `0.01` learning-rate prefactor, while both submitted `Train_revised*.ipynb` notebooks initialize Adam with `lr=1e-3`. The canonical configuration follows the executed notebooks and records `0.001`.

In [ ]:
from supgml.experiments import load_config, project_root

config = load_config(project_root() / "experiments/ch5_revised.json")
config


Run `supgml-train experiments/ch5_revised.json` in the DOLFINx environment. The package holds the repeated optimizer/checkpoint loop; notebook 06 retains the SPDE and reference-target definition that gives that loop its meaning.